# Gradient Debugging with LLM Hook Framework

This notebook shows how to debug training issues using gradient hooks.

## What You'll Learn
- Detecting vanishing/exploding gradients
- Tracking gradient flow through layers
- Identifying problematic layers
- Monitoring gradient norms during training
- Fixing common gradient issues

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from llm_hooks.core import HookManager
from llm_hooks.pytorch import BackwardHook
from llm_hooks.gradients import GradientTracker
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style('whitegrid')
%matplotlib inline

## Step 1: Create a Deep Network (Prone to Gradient Issues)

We'll create a deep network that might have gradient problems.

In [ ]:
class DeepNet(nn.Module):
    def __init__(self, input_size=128, hidden_size=256, num_layers=10, use_batchnorm=False):
        super().__init__()
        
        layers = []
        layers.append(nn.Linear(input_size, hidden_size))
        layers.append(nn.ReLU())
        
        for i in range(num_layers - 2):
            layers.append(nn.Linear(hidden_size, hidden_size))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(hidden_size))
            layers.append(nn.ReLU())
        
        layers.append(nn.Linear(hidden_size, 10))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# Create model WITHOUT BatchNorm (more likely to have gradient issues)
model = DeepNet(input_size=128, hidden_size=256, num_layers=10, use_batchnorm=False)
print(f"Created deep network with {len(list(model.parameters()))} parameter tensors")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 2: Set Up Gradient Tracking

We'll use both `BackwardHook` and `GradientTracker` to monitor gradients.

In [ ]:
# Create hook manager
manager = HookManager(name="gradient_debugging")

# Register backward hook to track layer-wise gradients
backward_hook = BackwardHook(
    compute_stats=True,
    detect_vanishing=True,
    detect_exploding=True,
    vanishing_threshold=1e-6,
    exploding_threshold=100.0,
)

# Register gradient tracker for parameter-wise tracking
gradient_tracker = GradientTracker(
    track_norm=True,
    track_distribution=True,
)

manager.register(backward_hook)
manager.register(gradient_tracker)
manager.apply_to_model(model)

print("✓ Gradient monitoring enabled")

## Step 3: Create Training Data

In [ ]:
# Create dummy dataset
batch_size = 32
num_batches = 5

train_data = [
    (torch.randn(batch_size, 128), torch.randint(0, 10, (batch_size,)))
    for _ in range(num_batches)
]

print(f"Created {num_batches} batches of training data")

## Step 4: Train and Monitor Gradients

Let's train for a few iterations while monitoring gradients.

In [ ]:
# Setup training
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
model.train()
losses = []

for batch_idx, (data, target) in enumerate(train_data):
    # Forward pass
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    
    # Backward pass (hooks will capture gradients)
    loss.backward()
    
    # Optimizer step
    optimizer.step()
    
    losses.append(loss.item())
    print(f"Batch {batch_idx + 1}/{num_batches}, Loss: {loss.item():.4f}")

print("\n✓ Training complete, gradient data captured")

## Step 5: Analyze Gradient Flow

Let's examine the captured gradient information.

In [ ]:
# Get results
results = manager.get_results()

print(f"Total gradient results: {len(results.results)}\n")

# Count issues
vanishing_count = 0
exploding_count = 0
nan_count = 0

for result in results.results:
    if result.hook_type == 'backward' and 'issues' in result.data:
        issues = result.data['issues']
        if 'vanishing_gradient' in issues:
            vanishing_count += 1
            print(f"⚠️  Vanishing gradient in {result.layer_name}")
        if 'exploding_gradient' in issues:
            exploding_count += 1
            print(f"⚠️  Exploding gradient in {result.layer_name}")
        if 'nan_gradient' in issues:
            nan_count += 1
            print(f"❌ NaN gradient in {result.layer_name}")

print(f"\nSummary:")
print(f"  Vanishing gradients: {vanishing_count}")
print(f"  Exploding gradients: {exploding_count}")
print(f"  NaN gradients: {nan_count}")

## Step 6: Visualize Gradient Norms by Layer

Let's plot gradient norms across layers to identify bottlenecks.

In [ ]:
# Extract gradient norms per layer
layer_grad_norms = {}

for result in results.results:
    if result.hook_type == 'backward' and 'stats' in result.data:
        layer_name = result.layer_name
        grad_norm = result.data['stats'].get('norm', 0)
        
        if layer_name not in layer_grad_norms:
            layer_grad_norms[layer_name] = []
        layer_grad_norms[layer_name].append(grad_norm)

# Calculate mean gradient norm per layer
layer_names = []
mean_norms = []

for layer_name, norms in sorted(layer_grad_norms.items()):
    layer_names.append(layer_name.split('.')[-1][:20])  # Truncate long names
    mean_norms.append(np.mean(norms))

# Plot gradient norms
plt.figure(figsize=(14, 6))
bars = plt.bar(range(len(layer_names)), mean_norms, color='steelblue', alpha=0.7)

# Highlight problematic layers
for i, norm in enumerate(mean_norms):
    if norm < 1e-6:
        bars[i].set_color('red')  # Vanishing
    elif norm > 100:
        bars[i].set_color('orange')  # Exploding

plt.xlabel('Layer')
plt.ylabel('Mean Gradient Norm')
plt.title('Gradient Flow Across Layers\n(Red=Vanishing, Orange=Exploding)')
plt.xticks(range(len(layer_names)), layer_names, rotation=45, ha='right')
plt.yscale('log')  # Log scale to see small values
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Step 7: Track Gradient Evolution During Training

Let's see how gradients change over training steps.

In [ ]:
# Group results by training step (batch)
gradients_by_step = {}

for result in results.results:
    if result.hook_type == 'gradient' and 'gradient_norm' in result.data:
        step = result.timestamp  # Use timestamp as proxy for step
        param_name = result.data.get('parameter_name', 'unknown')
        grad_norm = result.data['gradient_norm']
        
        if param_name not in gradients_by_step:
            gradients_by_step[param_name] = []
        gradients_by_step[param_name].append(grad_norm)

# Plot gradient evolution for a few parameters
if gradients_by_step:
    plt.figure(figsize=(12, 6))
    
    # Plot first 5 parameters
    for i, (param_name, norms) in enumerate(list(gradients_by_step.items())[:5]):
        plt.plot(norms, label=param_name[:30], marker='o', markersize=4)
    
    plt.xlabel('Training Step')
    plt.ylabel('Gradient Norm')
    plt.title('Gradient Norm Evolution During Training')
    plt.legend(loc='best', fontsize=8)
    plt.yscale('log')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No gradient tracking data available")

## Step 8: Solutions for Gradient Issues

Let's try fixing gradient issues with BatchNorm.

In [ ]:
# Create improved model WITH BatchNorm
model_improved = DeepNet(input_size=128, hidden_size=256, num_layers=10, use_batchnorm=True)

# Create new manager for comparison
with HookManager(name="improved_model") as improved_manager:
    improved_backward_hook = BackwardHook(
        compute_stats=True,
        detect_vanishing=True,
        detect_exploding=True,
    )
    
    improved_manager.register(improved_backward_hook)
    improved_manager.apply_to_model(model_improved)
    
    # Train for a few batches
    optimizer_improved = optim.Adam(model_improved.parameters(), lr=0.001)
    model_improved.train()
    
    for data, target in train_data[:3]:
        optimizer_improved.zero_grad()
        output = model_improved(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer_improved.step()
    
    # Check results
    improved_results = improved_manager.get_results()
    
    improved_vanishing = sum(
        1 for r in improved_results.results 
        if r.hook_type == 'backward' and 'vanishing_gradient' in r.data.get('issues', [])
    )
    
    print(f"\n{'='*60}")
    print("COMPARISON:")
    print(f"  Original model - Vanishing gradients: {vanishing_count}")
    print(f"  Improved model - Vanishing gradients: {improved_vanishing}")
    print(f"\n✓ BatchNorm reduced vanishing gradients!")

## Gradient Issue Solutions

### Vanishing Gradients
**Symptoms**: Gradient norms < 1e-6, early layers don't update

**Solutions**:
1. Use BatchNorm or LayerNorm
2. Use ReLU instead of sigmoid/tanh
3. Reduce network depth
4. Use residual connections
5. Initialize weights carefully (Xavier/He initialization)

### Exploding Gradients
**Symptoms**: Gradient norms > 100, NaN losses

**Solutions**:
1. Use gradient clipping: `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)`
2. Reduce learning rate
3. Use BatchNorm
4. Check for bugs in loss calculation

### NaN Gradients
**Symptoms**: NaN in gradients or losses

**Solutions**:
1. Add numerical stability (eps in log/sqrt)
2. Use gradient clipping
3. Check for division by zero
4. Reduce learning rate

### Example: Gradient Clipping

In [ ]:
# Training with gradient clipping
max_grad_norm = 1.0

for data, target in train_data[:2]:
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    loss.backward()
    
    # Clip gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
    
    optimizer.step()
    print(f"Loss: {loss.item():.4f} (with gradient clipping)")

## Best Practices for Gradient Monitoring

1. **Monitor early in training**: Catch issues before wasting compute
2. **Check all layers**: Issues often in early/deep layers
3. **Use log scale**: Gradients span many orders of magnitude
4. **Track over time**: See if issues persist or appear during training
5. **Compare architectures**: Test different configurations

## Next Steps

Try these exercises:

1. Apply to your own model architecture
2. Compare gradient flow with different activation functions
3. Test different normalization techniques
4. Monitor gradients during long training runs

**Next notebook**: `04_model_pruning.ipynb` - Use Fisher information for pruning